#### Question 1

In [2]:
# Create a dataset manually

import pandas as pd
import numpy as np

# Number of records
num_records = 100

# Generate synthetic data
np.random.seed(42) # for reproducibility

study_hours = np.random.uniform(10, 100, num_records).round(1)
attendance = np.random.uniform(50, 100, num_records).round(1) # percentage
assignment_score = np.random.uniform(40, 100, num_records).round(1)
previous_gpa = np.random.uniform(1.5, 4.0, num_records).round(2)

# Determine Pass/Fail based on a simple heuristic for demonstration
# A student is more likely to pass with higher study hours, attendance, assignment score, and GPA

# Weights
w_study = 0.2
w_attendance = 0.3
w_assignment = 0.3
w_gpa = 0.2

# Normalize features to a 0-1 scale for a more balanced weighted sum
study_hours_norm = (study_hours - study_hours.min()) / (study_hours.max() - study_hours.min())
attendance_norm = (attendance - attendance.min()) / (attendance.max() - attendance.min())
assignment_score_norm = (assignment_score - assignment_score.min()) / (assignment_score.max() - assignment_score.min())
previous_gpa_norm = (previous_gpa - previous_gpa.min()) / (previous_gpa.max() - previous_gpa.min())

weighted_sum = (w_study * study_hours_norm +
                w_attendance * attendance_norm +
                w_assignment * assignment_score_norm +
                w_gpa * previous_gpa_norm)

# threshold for passing
pass_threshold = 0.6

pass_fail = np.where(weighted_sum >= pass_threshold, 'Pass', 'Fail')

# Create DataFrame
data = pd.DataFrame({
    'Study Hours': study_hours,
    'Attendance (%)': attendance,
    'Assignment Score': assignment_score,
    'Previous GPA': previous_gpa,
    'Pass/Fail': pass_fail
})

print("Generated Dataset:")
print(data.head())
print(f"\nDataset shape: {data.shape}")
print(f"\nPass/Fail distribution:\n{data['Pass/Fail'].value_counts()}")

# Store the dataset in a variable that can be accessed later
df_students = data

Generated Dataset:
   Study Hours  Attendance (%)  Assignment Score  Previous GPA Pass/Fail
0         43.7            51.6              78.5          1.63      Fail
1         95.6            81.8              45.0          2.83      Fail
2         75.9            65.7              49.7          2.85      Fail
3         63.9            75.4              93.9          3.09      Pass
4         24.0            95.4              76.4          3.32      Pass

Dataset shape: (100, 5)

Pass/Fail distribution:
Pass/Fail
Fail    78
Pass    22
Name: count, dtype: int64


In [19]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

Q1_x, Q1_y = df_students.drop('Pass/Fail', axis=1), df_students['Pass/Fail']
x_train, x_test, y_train, y_test = train_test_split(Q1_x, Q1_y, test_size = 0.2, random_state = 42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model = LogisticRegression()
model.fit(x_train, y_train)
print("Moded Trained Successfully!\n")

y_pred = model.predict(x_test)

print("Testing wiht input: ", x_test[0])
print("Predicted Output: ", y_pred[0])


accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy: ", accuracy)

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix: \n", cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Moded Trained Successfully!

Testing wiht input:  [-1.38998987  1.29779244 -1.45102247  1.29906069]
Predicted Output:  Fail

Accuracy:  0.95

Confusion Matrix: 
 [[15  0]
 [ 1  4]]

Classification Report:
              precision    recall  f1-score   support

        Fail       0.94      1.00      0.97        15
        Pass       1.00      0.80      0.89         5

    accuracy                           0.95        20
   macro avg       0.97      0.90      0.93        20
weighted avg       0.95      0.95      0.95        20



#### Question 2

In [22]:
# Create a dataset manually

import pandas as pd
import numpy as np

# Number of records
num_records = 100

np.random.seed(43) # for reproducibility

# Generate synthetic data for email features
email_length = np.random.randint(50, 2000, num_records) # Email length in characters
num_links = np.random.randint(0, 15, num_records) # Number of links in email
num_special_chars = np.random.randint(1, 30, num_records) # Number of special characters
contains_free = np.random.choice(['Yes', 'No'], num_records, p=[0.3, 0.7]) # 30% chance of 'Free'

# Determine 'Spam/Not Spam' based on a heuristic
# Spam emails often have: shorter length, more links, more special characters, and contain 'Free'

# Normalize features to 0-1 scale for a weighted sum
email_length_norm = (email_length - email_length.min()) / (email_length.max() - email_length.min())
num_links_norm = (num_links - num_links.min()) / (num_links.max() - num_links.min())
num_special_chars_norm = (num_special_chars - num_special_chars.min()) / (num_special_chars.max() - num_special_chars.min())

# Convert 'Contains "Free"' to a numerical value (1 for Yes, 0 for No)
contains_free_numeric = np.where(contains_free == 'Yes', 1, 0)

# Assign weights to features (these can be tuned)
w_length = -0.2 # Shorter emails might be more spammy
w_links = 0.4   # More links, more spammy
w_special_chars = 0.3 # More special chars, more spammy
w_free = 0.3    # Contains 'Free', more spammy

weighted_spam_score = (w_length * (1 - email_length_norm) + # Invert length as shorter is more spammy
                       w_links * num_links_norm +
                       w_special_chars * num_special_chars_norm +
                       w_free * contains_free_numeric)

# Define a threshold for classifying as Spam
spam_threshold = 0.5

spam_not_spam = np.where(weighted_spam_score >= spam_threshold, 'Spam', 'Not Spam')

# Create DataFrame
df_emails = pd.DataFrame({
    'Email Length': email_length,
    'Number of Links': num_links,
    'Number of Special Characters': num_special_chars,
    'Contains "Free"': contains_free,
    'Spam/Not Spam': spam_not_spam
})

print("Generated Email Dataset:")
print(df_emails.head())
print(f"\nDataset shape: {df_emails.shape}")
print(f"\nSpam/Not Spam distribution:\n{df_emails['Spam/Not Spam'].value_counts()}")

Generated Email Dataset:
   Email Length  Number of Links  Number of Special Characters  \
0          1910                1                            17   
1          1394                6                            29   
2           305                0                            22   
3          1891                0                            14   
4           327               12                            17   

  Contains "Free" Spam/Not Spam  
0             Yes      Not Spam  
1              No      Not Spam  
2              No      Not Spam  
3             Yes      Not Spam  
4              No      Not Spam  

Dataset shape: (100, 5)

Spam/Not Spam distribution:
Spam/Not Spam
Not Spam    80
Spam        20
Name: count, dtype: int64


In [71]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd


# Change Yes No to 0 1
df_emails['Spam/Not Spam'] = df_emails['Spam/Not Spam'].replace({'Spam': 1, 'Not Spam': 0}).astype(int)
df_emails['Contains "Free"'] = df_emails['Contains "Free"'].replace({'Yes': 1, 'No': 0}).astype(int)


Q2_x, Q2_y = df_emails.drop('Spam/Not Spam', axis=1), df_emails['Spam/Not Spam']
x_train, x_test, y_train, y_test = train_test_split(Q2_x, Q2_y, test_size=0.2, random_state=42)


scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

knn = KNeighborsClassifier()
knn.fit(x_train_scaled, y_train)
print("KNN Model Trained Successfully!\n")

model = LogisticRegression()
model.fit(x_train_scaled, y_train)
print("Logistic Model Trained Successfully!\n")

y_pred_knn = knn.predict(x_test_scaled)
y_pred_log = model.predict(x_test_scaled)

knn_acc = accuracy_score(y_test, y_pred_knn)
log_acc = accuracy_score(y_test, y_pred_log)

print("Testing with input:\n", x_test_scaled[1])

print("\nPredicted Outputs (0 for Not Spam, 1 for Spam):")
print("KNN: ", y_pred_knn[1])
print("Logistic Regression: ", y_pred_log[1])

print("\nKNN Accuracy: ", knn_acc)
print("Logistic Regression Accuracy: ", log_acc)

if knn_acc == log_acc:
    print("\nBoth models have the same accuracy.")
elif knn_acc > log_acc:
    print("\nKNN model has a better accuracy.")
else:
    print("\nLogistic Regression model has a better accuracy.")

KNN Model Trained Successfully!

Logistic Model Trained Successfully!

Testing with input:
 [-1.34158141  0.21841119  1.44278159 -0.55809982 -0.5       ]

Predicted Outputs (0 for Not Spam, 1 for Spam):
KNN:  0
Logistic Regression:  0

KNN Accuracy:  1.0
Logistic Regression Accuracy:  1.0

Both models have the same accuracy.


#### Question 3

In [73]:
# Create dataset manually

import pandas as pd
import numpy as np

# Number of records
num_records = 100

np.random.seed(44) # for reproducibility

# Generate synthetic data for house features
house_size = np.random.randint(800, 5000, num_records) # House Size in sq ft
num_rooms = np.random.randint(2, 10, num_records) # Number of Rooms
location_score = np.random.uniform(1, 10, num_records).round(1) # Location Score (e.g., proximity to amenities, schools)
age_of_house = np.random.randint(0, 70, num_records) # Age of House in years

# Determine 'House Price' based on a heuristic
# Higher house size, more rooms, higher location score, and newer house generally lead to higher prices

# Normalize features to 0-1 scale for a weighted sum
house_size_norm = (house_size - house_size.min()) / (house_size.max() - house_size.min())
num_rooms_norm = (num_rooms - num_rooms.min()) / (num_rooms.max() - num_rooms.min())
location_score_norm = (location_score - location_score.min()) / (location_score.max() - location_score.min())
age_of_house_norm = (age_of_house - age_of_house.min()) / (age_of_house.max() - age_of_house.min())

# Assign weights to features (these can be tuned)
w_size = 0.4
w_rooms = 0.2
w_location = 0.3
w_age = -0.1 # Negative weight for age (newer is better)

# Calculate a base price and add some noise
base_price = (w_size * house_size_norm + \
              w_rooms * num_rooms_norm + \
              w_location * location_score_norm + \
              w_age * (1 - age_of_house_norm)) * 500000 + 100000 # Scale to a realistic price range

# Add some random noise to the price
house_price = (base_price + np.random.normal(0, 50000, num_records)).round(0)

# Create DataFrame
df_houses = pd.DataFrame({
    'House Size (sq ft)': house_size,
    'Number of Rooms': num_rooms,
    'Location Score': location_score,
    'Age of House': age_of_house,
    'House Price ($)': house_price
})

print("Generated House Dataset:")
print(df_houses.head())
print(f"\nDataset shape: {df_houses.shape}")

Generated House Dataset:
   House Size (sq ft)  Number of Rooms  Location Score  Age of House  \
0                4291                8             8.7            20   
1                3601                9             3.8            34   
2                1997                4             5.4            62   
3                1371                6             4.6            11   
4                4771                2             9.9            28   

   House Price ($)  
0         405027.0  
1         336646.0  
2         278973.0  
3         262877.0  
4         392484.0  

Dataset shape: (100, 5)


In [82]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

Q3_x, Q3_y = df_houses.drop('House Price ($)', axis=1), df_houses['House Price ($)']
x_train, x_test, y_train, y_test = train_test_split(Q3_x, Q3_y, test_size = 0.2, random_state=42)

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model = LinearRegression()
model.fit(x_train, y_train)
print("Model Trained Successfully!\n")

y_pred = model.predict(x_test)

print("Testing for value: ", x_test[0])
print("Predicted Output: ", y_pred[0])
print("Actual Output: ", y_test[0])

mse = mean_squared_error(y_test, y_pred)
print("\nMean Squared Error:", mse)

rmse = np.sqrt(mse)
print("RMSE:", rmse)

r2 = r2_score(y_test, y_pred)
print("R-squared:", r2)

Model Trained Successfully!

Testing for value:  [ 1.13319055  1.48038214 -0.0264606   1.63574132]
Predicted Output:  435234.96603648196
Actual Output:  405027.0

Mean Squared Error: 3308440233.605735
RMSE: 57519.042356473
R-squared: 0.6943876398203426


#### Question 4

In [129]:
# Create a dataset manually

import pandas as pd
import numpy as np

# Number of records
num_records = 100

np.random.seed(45) # for reproducibility

# Generate synthetic data for employee features
years_of_experience = np.random.uniform(0, 30, num_records).round(1)
education_levels = ['High School', 'Bachelors', 'Masters', 'PhD']
education_level = np.random.choice(education_levels, num_records, p=[0.2, 0.4, 0.3, 0.1])
skill_score = np.random.uniform(1, 10, num_records).round(1)
age = np.random.randint(22, 65, num_records)

# Determine 'Salary' based on a heuristic
# Higher experience, education, skill, and age (within reason) generally lead to higher salaries

# Convert education level to numerical for weighting
education_map = {'High School': 1, 'Bachelors': 2, 'Masters': 3, 'PhD': 4}
education_numeric = np.array([education_map[level] for level in education_level])

# Normalize numerical features to a 0-1 scale
years_of_experience_norm = (years_of_experience - years_of_experience.min()) / (years_of_experience.max() - years_of_experience.min())
education_numeric_norm = (education_numeric - education_numeric.min()) / (education_numeric.max() - education_numeric.min())
skill_score_norm = (skill_score - skill_score.min()) / (skill_score.max() - skill_score.min())
age_norm = (age - age.min()) / (age.max() - age.min())

# Assign weights to features (these can be tuned)
w_experience = 0.4
w_education = 0.25
w_skill = 0.25
w_age = 0.1 # Slight positive correlation for age

# Calculate a base salary and add some noise
base_salary = (w_experience * years_of_experience_norm +
               w_education * education_numeric_norm +
               w_skill * skill_score_norm +
               w_age * age_norm) * 80000 + 40000 # Scale to a realistic salary range

# Add some random noise to the salary
salary = (base_salary + np.random.normal(0, 10000, num_records)).round(0)

# Create DataFrame
df_employees = pd.DataFrame({
    'Years of Experience': years_of_experience,
    'Education Level': education_level,
    'Skill Score': skill_score,
    'Age': age,
    'Salary ($)': salary
})

print("Generated Employee Dataset:")
print(df_employees.head())
print(f"\nDataset shape: {df_employees.shape}")

Generated Employee Dataset:
   Years of Experience Education Level  Skill Score  Age  Salary ($)
0                 29.7       Bachelors          7.2   51     91333.0
1                 16.5       Bachelors          6.7   53     82777.0
2                  8.4       Bachelors          6.9   47     74140.0
3                  2.3       Bachelors          9.6   42     60358.0
4                 13.3     High School          9.0   35     68857.0

Dataset shape: (100, 5)


In [130]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
import pandas as pd

print("We see we have 4 Education Levels we need to convert to numeric:")
education_levels = df_employees['Education Level'].unique()
print(education_levels)

df_employees['Education Level'] = df_employees['Education Level'].replace({'High School': 1, 'Bachelors' : 2, 'Masters' : 3, 'PhD' : 4}).astype(int)

print("\nEducation Levels after conversion:")
education_levels = df_employees['Education Level'].unique()
print(education_levels)

Q4_x, Q4_y = df_employees.drop('Salary ($)', axis=1), df_employees['Salary ($)']
x_train, x_test, y_train, y_test = train_test_split(Q4_x, Q4_y, test_size = 0.2, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

model = LinearRegression()
model.fit(x_train, y_train)
print("\nLinear Regression Model Trained Successfully!")

y_pred = model.predict(x_test)

print("\nEvaluation Metrics: ")
print("Testing for value: ", x_test[0])
print("Predicted Output: ", y_pred[0])
print("Actual Output: ", y_test[0])

r2_reg = r2_score(y_test, y_pred)
print("R2 Score:", r2_reg)

tree_model = DecisionTreeRegressor(random_state=42)
tree_model.fit(x_train, y_train)
print("\nDecision Tree Model Trained Successfully!")

y_pred_tree = tree_model.predict(x_test)
print("\nEvaluation Metrics: ")
print("Testing for value: ", x_test[0])
print("Predicted Output: ", y_pred_tree[0])
print("Actual Output: ", y_test[0])

r2_tree = r2_score(y_test, y_pred_tree)
print("R2 Score:", r2_tree)

if r2_reg > r2_tree:
  print("\nLinear Regression Model is the best")
else:
  print("\nDecision Tree Model is the best\n")

We see we have 4 Education Levels we need to convert to numeric:
['Bachelors' 'High School' 'PhD' 'Masters']

Education Levels after conversion:
[2 1 4 3]

Linear Regression Model Trained Successfully!

Evaluation Metrics: 
Testing for value:  [ 0.24799055 -1.38675049 -1.64537481  0.11424339]
Predicted Output:  66401.48040612126
Actual Output:  91333.0
R2 Score: 0.6792878244298934

Decision Tree Model Trained Successfully!

Evaluation Metrics: 
Testing for value:  [ 0.24799055 -1.38675049 -1.64537481  0.11424339]
Predicted Output:  79250.0
Actual Output:  91333.0
R2 Score: 0.2520057005867309

Linear Regression Model is the best


/tmp/ipykernel_1249/3112805996.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_employees['Education Level'] = df_employees['Education Level'].replace({'High School': 1, 'Bachelors' : 2, 'Masters' : 3, 'PhD' : 4}).astype(int)


#### Question 5

In [132]:
# Create a dataset manually

import pandas as pd
import numpy as np

# Number of records
num_records = 100

np.random.seed(46) # for reproducibility

# Generate synthetic data for customer features
age = np.random.randint(18, 70, num_records)
income = np.random.uniform(20000, 150000, num_records).round(2)
previous_purchases = np.random.randint(0, 50, num_records)
time_on_website = np.random.uniform(5, 60, num_records).round(2) # minutes

# Determine 'Buy/Not Buy' based on a heuristic
# Customers are more likely to buy if they have higher income, more previous purchases,
# and spend more time on the website. Age might have a non-linear or less direct impact.

# Normalize features to 0-1 scale for a weighted sum
income_norm = (income - income.min()) / (income.max() - income.min())
previous_purchases_norm = (previous_purchases - previous_purchases.min()) / (previous_purchases.max() - previous_purchases.min())
time_on_website_norm = (time_on_website - time_on_website.min()) / (time_on_website.max() - time_on_website.min())

# Assign weights to features (these can be tuned)
w_income = 0.4
w_previous_purchases = 0.3
w_time_on_website = 0.3

weighted_buy_score = (w_income * income_norm +
                      w_previous_purchases * previous_purchases_norm +
                      w_time_on_website * time_on_website_norm)

# Define a threshold for classifying as 'Buy'
buy_threshold = 0.5

buy_not_buy = np.where(weighted_buy_score >= buy_threshold, 'Buy', 'Not Buy')

# Create DataFrame
df_customers = pd.DataFrame({
    'Age': age,
    'Income': income,
    'Previous Purchases': previous_purchases,
    'Time Spent on Website': time_on_website,
    'Buy/Not Buy': buy_not_buy
})

print("Generated Customer Dataset:")
print(df_customers.head())
print(f"\nDataset shape: {df_customers.shape}")
print(f"\nBuy/Not Buy distribution:\n{df_customers['Buy/Not Buy'].value_counts()}")

Generated Customer Dataset:
   Age     Income  Previous Purchases  Time Spent on Website Buy/Not Buy
0   23  143061.14                  11                  59.89         Buy
1   58   64954.45                  27                  28.84     Not Buy
2   37   72952.76                   0                   6.88     Not Buy
3   36  127760.44                  11                   8.26     Not Buy
4   29  112822.00                  32                  53.62         Buy

Dataset shape: (100, 5)

Buy/Not Buy distribution:
Buy/Not Buy
Not Buy    61
Buy        39
Name: count, dtype: int64


In [136]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd


# Change Buy Not Buy to 0 1
df_customers['Buy/Not Buy'] = df_customers['Buy/Not Buy'].replace({'Buy': 1, 'Not Buy': 0}).astype(int)

Q5_x, Q5_y = df_customers.drop('Buy/Not Buy', axis=1), df_customers['Buy/Not Buy']
x_train, x_test, y_train, y_test = train_test_split(Q5_x, Q5_y, test_size=0.2, random_state=42)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

knn = KNeighborsClassifier()
knn.fit(x_train_scaled, y_train)
print("KNN Model Trained Successfully!\n")

model = LogisticRegression()
model.fit(x_train_scaled, y_train)
print("Logistic Model Trained Successfully!\n")

y_pred_knn = knn.predict(x_test_scaled)
y_pred_log = model.predict(x_test_scaled)

knn_acc = accuracy_score(y_test, y_pred_knn)
log_acc = accuracy_score(y_test, y_pred_log)

print("Testing with input:\n", x_test_scaled[1])

print("\nPredicted Outputs (0 for Not Buy, 1 for Buy):")
print("KNN: ", y_pred_knn[1])
print("Logistic Regression: ", y_pred_log[1])

print("Actual Output: ", y_test.iloc[1])

print("\nKNN Accuracy: ", knn_acc)
print("Logistic Regression Accuracy: ", log_acc)

if knn_acc == log_acc:
    print("\nBoth models have the same accuracy.")
elif knn_acc > log_acc:
    print("\nKNN model has a better accuracy.")
else:
    print("\nLogistic Regression model has a better accuracy.")

KNN Model Trained Successfully!

Logistic Model Trained Successfully!

Testing with input:
 [ 1.71847437  0.62414031  0.63350031 -1.58860452]

Predicted Outputs (0 for Not Buy, 1 for Buy):
KNN:  1
Logistic Regression:  0
Actual Output:  0

KNN Accuracy:  0.85
Logistic Regression Accuracy:  0.9

Logistic Regression model has a better accuracy.
